In [2]:
print("HELLO")

HELLO


In [3]:
import pandas as pd
import numpy as np

import re
import unicodedata
from collections import defaultdict, Counter
from pathlib import Path

from anyascii import anyascii
from rapidfuzz.distance import Levenshtein

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("sampled_data")

S1_FILE = BASE_DIR / "sample_source1.tsv"
S2_FILE = BASE_DIR / "sample_source2.tsv"
S3_FILE = BASE_DIR / "sample_source3.tsv"
GT_FILE = BASE_DIR / "sample_ground_truth.tsv"

OUTPUT_DIR = Path("blocking_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADDRESS_OVERLAP_THRESHOLD = 2

# Very common address words should not drive blocking.
WEAK_ADDRESS_TOKENS = {
    "road", "rd",
    "street", "st",
    "avenue", "ave",
    "lane", "ln",
    "drive", "dr",
    "highway", "hwy",
    "boulevard", "blvd",
    "way",
    "place", "pl",
    "parkway", "pkwy",
    "unit",
    "suite", "ste",
    "apt", "apartment",
    "floor",
    "fl",
    "building", "bldg",
    "block",
    "district",
    "city",
    "state",
    "county",
    "india",
    "usa",
    "us",
}

MIN_TOKEN_LENGTH = 2

print("Configuration loaded.")
print(f"Address overlap threshold: {ADDRESS_OVERLAP_THRESHOLD}")
print(f"Weak address tokens: {len(WEAK_ADDRESS_TOKENS)}")

Configuration loaded.
Address overlap threshold: 2
Weak address tokens: 36


In [4]:
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

s1 = pd.read_csv(S1_FILE, sep="\t", dtype=str).fillna("")
s2 = pd.read_csv(S2_FILE, sep="\t", dtype=str).fillna("")
s3 = pd.read_csv(S3_FILE, sep="\t", dtype=str).fillna("")
gt = pd.read_csv(GT_FILE, sep="\t", dtype=str).fillna("")

print(f"S1 shape: {s1.shape}")
print(f"S2 shape: {s2.shape}")
print(f"S3 shape: {s3.shape}")
print(f"GT shape: {gt.shape}")

print("\nColumns:")
print("S1:", list(s1.columns))
print("S2:", list(s2.columns))
print("S3:", list(s3.columns))
print("GT:", list(gt.columns))


LOADING DATA
S1 shape: (1000, 4)
S2 shape: (9864, 4)
S3 shape: (10842, 4)
GT shape: (1000, 2)

Columns:
S1: ['entity_id', 'business_name', 'business_address', 'country']
S2: ['entity_id', 'business_name', 'business_address', 'country']
S3: ['entity_id', 'business_name', 'business_address', 'country']
GT: ['source1_entity_id', 'matched_entity_ids']


In [5]:
def is_latin_char(ch):
    """
    Returns True if the character belongs to the Latin script.
    """
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return False

    return "LATIN" in name


def contains_non_latin(text):
    """
    Detect whether text contains alphabetic characters
    outside the Latin script.
    """
    for ch in text:
        if ch.isalpha() and not is_latin_char(ch):
            return True
    return False


def transliterate_text(text):
    """
    Convert non-Latin text into a Latin-oriented representation.

    Original text is preserved elsewhere.
    """
    if not text:
        return ""

    return anyascii(text)


def normalize_text(text, transliterate=False):
    """
    General normalization.

    Steps:
    - Unicode normalization
    - lowercase
    - optional transliteration
    - punctuation -> spaces
    - collapse whitespace
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()

    if transliterate:
        text = transliterate_text(text)

    # Keep letters and numbers, replace everything else with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    """
    Basic normalized whitespace tokenization.
    """
    if not text:
        return []

    return text.split()


def normalize_token(token):
    return normalize_text(token, transliterate=False)


def transliterated_tokens(text):
    """
    Tokenize first, then transliterate tokens that contain
    non-Latin characters.
    """
    result = []

    for token in tokenize(normalize_text(text)):
        if contains_non_latin(token):
            token = transliterate_text(token)

        token = normalize_text(token, transliterate=False)

        if token:
            result.append(token)

    return result

In [6]:
def prepare_source(df):
    df = df.copy()

    df["name_norm"] = df["business_name"].apply(
        lambda x: normalize_text(x, transliterate=False)
    )

    df["address_norm"] = df["business_address"].apply(
        lambda x: normalize_text(x, transliterate=False)
    )

    df["name_translit"] = df["business_name"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    df["address_translit"] = df["business_address"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    df["country_norm"] = df["country"].apply(
        lambda x: normalize_text(x, transliterate=True)
    )

    return df


print("Preparing normalized representations...")

s1 = prepare_source(s1)
s2 = prepare_source(s2)
s3 = prepare_source(s3)

print("Normalization complete.")

Preparing normalized representations...
Normalization complete.


In [7]:
non_latin_s2 = s2[
    s2["business_name"].apply(contains_non_latin) |
    s2["business_address"].apply(contains_non_latin)
]

non_latin_s3 = s3[
    s3["business_name"].apply(contains_non_latin) |
    s3["business_address"].apply(contains_non_latin)
]

print(f"S2 records containing non-Latin text: {len(non_latin_s2)}")
print(f"S3 records containing non-Latin text: {len(non_latin_s3)}")

print("\nS2 transliteration examples:")
print(
    non_latin_s2[
        ["business_name", "name_translit",
         "business_address", "address_translit"]
    ].head(10).to_string(index=False)
)

S2 records containing non-Latin text: 1598
S3 records containing non-Latin text: 1389

S2 transliteration examples:
                               business_name                           name_translit                                                                                                     business_address                                                                                              address_translit
                பிளாக் ஐடி பிரைவேட் லிமிடெட்             pilak aiti piraivet limitet                                                                                 SF.NO364/IB2, COIMBATORE, Tamil Nadu                                                                            sf no364 ib2 coimbatore tamil nadu
      नॉर्थ इंफ्रा ट्रेडिंग प्राइवेट लिमिटेड    north imphra tredimg praivet limited                                                                             NO. 634 UNIT NO.- 938, WEST DELHI, Delhi                                                           

In [8]:
def get_informative_address_tokens(address):
    """
    Extract address tokens that are useful for blocking.

    Uses transliterated representation so non-Latin addresses
    can participate in the same Latin-oriented index.
    """

    tokens = transliterated_tokens(address)

    informative = set()

    for token in tokens:

        if len(token) < MIN_TOKEN_LENGTH:
            continue

        if token in WEAK_ADDRESS_TOKENS:
            continue

        # Ignore tokens that are purely punctuation/noise
        if not re.search(r"[a-z0-9]", token):
            continue

        informative.add(token)

    return informative

In [9]:
s1["address_tokens"] = s1["business_address"].apply(
    get_informative_address_tokens
)

s2["address_tokens"] = s2["business_address"].apply(
    get_informative_address_tokens
)

s3["address_tokens"] = s3["business_address"].apply(
    get_informative_address_tokens
)

print("Example S1 address tokens:")

for _, row in s1.head(10).iterrows():
    print(row["entity_id"])
    print(row["address_tokens"])
    print()

Example S1 address tokens:
S1-53356671
{'20', '4850', 'otisco', 'ny'}

S1-320151505
{'woodburn', 'in', '19034'}

S1-938947364
{'az', 'mesa', 'osage', '7241'}

S1-195839862
{'elakamon', '606', 'trivandrum', 'thiruvananthapuram', 'prabhul', 'cottage', 'kerala', 'no', 'ayiroor', 'karimbalur'}

S1-655046555
{'tn', '4255', 'charleswood', 'memphis'}

S1-758070915
{'al', '1641', 'virginia', 'hueytown'}

S1-169338169
{'31', 'me', 'guillemette', 'sanford'}

S1-796421498
{'billabong', '1204', '12thfloor', 'roya', 'malad', 'nr', 'oasis', 'nagarmalvani', 'jankalyan', 'maharashtra', 'mumbai', 'school'}

S1-802535179
{'wood', 'cedar', 'tx', 'cove', 'ridge', 'park', '2105'}

S1-97176033
{'ramdev', 'maharashtra', 'chsl', 'thane', 'sai', 'dham', 'park', 'new', '301'}



In [10]:
def build_address_index(df):
    index = defaultdict(set)

    for _, row in df.iterrows():

        entity_id = row["entity_id"]

        for token in row["address_tokens"]:
            index[token].add(entity_id)

    return index


print("=" * 70)
print("BUILDING ADDRESS INDEX")
print("=" * 70)

s2_address_index = build_address_index(s2)
s3_address_index = build_address_index(s3)

print(f"S2 address index tokens: {len(s2_address_index)}")
print(f"S3 address index tokens: {len(s3_address_index)}")

BUILDING ADDRESS INDEX
S2 address index tokens: 16454
S3 address index tokens: 16573


In [11]:
def build_entity_lookup(df):
    return df.set_index("entity_id").to_dict("index")


s2_lookup = build_entity_lookup(s2)
s3_lookup = build_entity_lookup(s3)


def generate_address_candidates_for_s1(
    s1_row,
    address_index,
    entity_lookup,
    source_name
):
    """
    Generate address-block candidates for one S1 record.
    """

    s1_country = s1_row["country_norm"]
    s1_tokens = s1_row["address_tokens"]

    token_counts = Counter()

    for token in s1_tokens:

        candidate_ids = address_index.get(token, set())

        for candidate_id in candidate_ids:
            token_counts[candidate_id] += 1

    candidates = []

    for candidate_id, overlap_count in token_counts.items():

        candidate = entity_lookup[candidate_id]

        if candidate["country_norm"] != s1_country:
            continue

        if overlap_count < ADDRESS_OVERLAP_THRESHOLD:
            continue

        candidates.append({
            "s1_entity_id": s1_row["entity_id"],
            "candidate_entity_id": candidate_id,
            "candidate_source": source_name,
            "block_type": "address",
            "shared_address_tokens": overlap_count
        })

    return candidates

In [12]:
print("=" * 70)
print("RUNNING ADDRESS BLOCKER")
print("=" * 70)

candidate_rows = []

for i, (_, row) in enumerate(s1.iterrows(), start=1):

    if i % 100 == 0 or i == 1:
        print(f"Processing S1 {i}/{len(s1)}")

    # S2
    candidate_rows.extend(
        generate_address_candidates_for_s1(
            row,
            s2_address_index,
            s2_lookup,
            "S2"
        )
    )

    # S3
    candidate_rows.extend(
        generate_address_candidates_for_s1(
            row,
            s3_address_index,
            s3_lookup,
            "S3"
        )
    )

address_candidates = pd.DataFrame(candidate_rows)

print("\nAddress blocking complete.")
print(f"Candidate rows: {len(address_candidates):,}")

if len(address_candidates):
    print(
        "Unique S1 entities with candidates:",
        address_candidates["s1_entity_id"].nunique()
    )

RUNNING ADDRESS BLOCKER
Processing S1 1/1000
Processing S1 100/1000
Processing S1 200/1000
Processing S1 300/1000
Processing S1 400/1000
Processing S1 500/1000
Processing S1 600/1000
Processing S1 700/1000
Processing S1 800/1000
Processing S1 900/1000
Processing S1 1000/1000

Address blocking complete.
Candidate rows: 186,946
Unique S1 entities with candidates: 993


In [13]:
address_candidates = (
    address_candidates
    .drop_duplicates(
        subset=[
            "s1_entity_id",
            "candidate_entity_id"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Unique S1-candidate pairs: "
    f"{len(address_candidates):,}"
)

Unique S1-candidate pairs: 186,946


In [14]:
def parse_ground_truth(gt_df):
    rows = []

    for _, row in gt_df.iterrows():

        s1_id = row["source1_entity_id"]
        matched = str(row["matched_entity_ids"]).strip()

        if not matched:
            continue

        for candidate_id in matched.split(","):

            candidate_id = candidate_id.strip()

            if candidate_id:
                rows.append({
                    "s1_entity_id": s1_id,
                    "true_entity_id": candidate_id
                })

    return pd.DataFrame(rows)


gt_pairs = parse_ground_truth(gt)

print(f"Ground-truth positive pairs: {len(gt_pairs):,}")

print(
    "Ground-truth S1 entities with at least one match:",
    gt_pairs["s1_entity_id"].nunique()
)

Ground-truth positive pairs: 3,451
Ground-truth S1 entities with at least one match: 934


In [15]:
candidate_pairs = address_candidates[
    [
        "s1_entity_id",
        "candidate_entity_id"
    ]
].drop_duplicates()

evaluation = gt_pairs.merge(
    candidate_pairs,
    left_on=[
        "s1_entity_id",
        "true_entity_id"
    ],
    right_on=[
        "s1_entity_id",
        "candidate_entity_id"
    ],
    how="left",
    indicator=True
)

evaluation["retrieved"] = (
    evaluation["_merge"] == "both"
)

retrieved = evaluation["retrieved"].sum()
total_true = len(evaluation)

blocking_recall = (
    retrieved / total_true
    if total_true > 0
    else 0
)

print("=" * 70)
print("BLOCKING RECALL")
print("=" * 70)

print(f"True matching pairs: {total_true:,}")
print(f"Retrieved by blocking: {retrieved:,}")
print(f"Missed true pairs: {total_true - retrieved:,}")
print(f"Blocking recall: {blocking_recall:.4%}")

BLOCKING RECALL
True matching pairs: 3,451
Retrieved by blocking: 3,268
Missed true pairs: 183
Blocking recall: 94.6972%


In [16]:
candidate_counts = (
    candidate_pairs
    .groupby("s1_entity_id")
    .size()
)

print("=" * 70)
print("CANDIDATE VOLUME")
print("=" * 70)

print(f"Total candidate pairs: {len(candidate_pairs):,}")

print(
    f"Average candidates/S1: "
    f"{candidate_counts.mean():.2f}"
)

print(
    f"Median candidates/S1: "
    f"{candidate_counts.median():.2f}"
)

print(
    f"P95 candidates/S1: "
    f"{candidate_counts.quantile(0.95):.2f}"
)

print(
    f"P99 candidates/S1: "
    f"{candidate_counts.quantile(0.99):.2f}"
)

print(
    f"Maximum candidates/S1: "
    f"{candidate_counts.max()}"
)

total_possible = len(s1) * (len(s2) + len(s3))

reduction_ratio = (
    1 - len(candidate_pairs) / total_possible
    if total_possible > 0
    else 0
)

print(f"\nTotal possible Cartesian pairs: {total_possible:,}")
print(f"Candidate reduction ratio: {reduction_ratio:.4%}")

CANDIDATE VOLUME
Total candidate pairs: 186,946
Average candidates/S1: 188.26
Median candidates/S1: 22.00
P95 candidates/S1: 876.80
P99 candidates/S1: 1399.36
Maximum candidates/S1: 1765

Total possible Cartesian pairs: 20,706,000
Candidate reduction ratio: 99.0971%


In [17]:
missed_pairs = evaluation[
    ~evaluation["retrieved"]
][
    [
        "s1_entity_id",
        "true_entity_id"
    ]
].copy()

print("=" * 70)
print("MISSED TRUE PAIRS")
print("=" * 70)

print(f"Missed pairs: {len(missed_pairs):,}")

missed_pairs.to_csv(
    OUTPUT_DIR / "missed_blocking_pairs.tsv",
    sep="\t",
    index=False
)

print(
    f"Saved to: "
    f"{OUTPUT_DIR / 'missed_blocking_pairs.tsv'}"
)

MISSED TRUE PAIRS
Missed pairs: 183
Saved to: blocking_results\missed_blocking_pairs.tsv


In [18]:
all_source_records = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
)

s1_lookup = s1.set_index("entity_id")
candidate_lookup = all_source_records.set_index("entity_id")


for _, pair in missed_pairs.head(20).iterrows():

    s1_id = pair["s1_entity_id"]
    true_id = pair["true_entity_id"]

    s1_row = s1_lookup.loc[s1_id]
    true_row = candidate_lookup.loc[true_id]

    print("=" * 70)
    print("MISSED PAIR")
    print("=" * 70)

    print("S1:", s1_id)
    print("Name:", s1_row["business_name"])
    print("Address:", s1_row["business_address"])
    print("Country:", s1_row["country"])

    print()

    print("TRUE:", true_id)
    print("Name:", true_row["business_name"])
    print("Address:", true_row["business_address"])
    print("Country:", true_row["country"])

    print()

MISSED PAIR
S1: S1-511076246
Name: Agra Granites
Address: Flat No-4, Sector-4B, Vaishno Plaza Awas Vikas Colony Sikandra, Agra, Uttar Pradesh
Country: India

TRUE: S2-100727491
Name: Agra Granites [Limited]
Address: 
Country: India

MISSED PAIR
S1: S1-210935903
Name: Leann Colon Federal Acquisitions Inc
Address: 9229 106th Way, Fl 1, Scottsdale, AZ
Country: US

TRUE: S3-121609463
Name: Leann Colon  Federal
Address: 
Country: US

MISSED PAIR
S1: S1-10580196
Name: Secure Telecom
Address: 310 Calion Street, Unit Apt C, Jonesboro, AR
Country: US

TRUE: S2-34357118
Name: SECURE TELECOM INC.
Address: 
Country: US

MISSED PAIR
S1: S1-748867572
Name: Britt Best Willow LLC
Address: 16955 Toronto Avenue, Unit Apartment 115, Prior Lake, MN
Country: US

TRUE: S3-726962279
Name: britt best willow willow llc
Address: 
Country: US

MISSED PAIR
S1: S1-30752064
Name: Rampur Machinery Limited
Address: 25 Tehsil Swar, Mohalla Kashipur Suar, Rampur, Uttar Pradesh
Country: India

TRUE: S3-997789709
Name: R